# Pandas 03 — Selecting, filtering and transforming

**What's in here**
- `loc` / `iloc` / `[]` — labels vs positions, inclusive slicing
- Boolean masks, `isin`, `between`, `query`, string filters, `select_dtypes`
- Renaming, `assign`, `pipe` for chained pipelines
- Vectorised ops vs `apply` (with timings), `map`, `np.where` / `np.select`
- Binning with `cut` / `qcut`, `rank`, `diff`, `pct_change`, `shift` (and its sign)
- `sort_values`, `nlargest`, `idxmax`, index handling
- The `SettingWithCopyWarning` and how to write assignments that never trigger it

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

In [2]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
df["hour"] = df["time"].dt.hour
df["dow"] = df["time"].dt.dayofweek
df.head(3)

,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,hour,dow
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83,0,5
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21,1,5
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71,2,5


## 1. `[]` — columns by name

`df["col"]` → Series. `df[["a", "b"]]` → DataFrame (note the double brackets). `df[mask]` with a boolean Series filters **rows**. That overloading is why `[]` is only for quick work.

In [3]:
print(type(df["temp_c"]), type(df[["temp_c", "wind_ms"]]))
df[["time", "consumption_mwh"]].head(3)

<class 'pandas.core.series.Series'> <class 'pandas.core.frame.DataFrame'>


,time,consumption_mwh
0,2022-01-01 00:00:00+00:00,26858.4
1,2022-01-01 01:00:00+00:00,26177.8
2,2022-01-01 02:00:00+00:00,26229.4


## 2. `loc` — by label, `iloc` — by position

`loc[rows, cols]` uses index labels and column names; slices are **inclusive** on both ends. `iloc[rows, cols]` uses integer positions; slices are half-open like Python lists.

In [4]:
print(df.loc[0:2, ["time", "temp_c"]])          # rows 0,1,2 (inclusive!)
print(df.iloc[0:2, [0, 2]])                      # rows 0,1 only

                       time  temp_c
0 2022-01-01 00:00:00+00:00    0.11
1 2022-01-01 01:00:00+00:00   -0.18
2 2022-01-01 02:00:00+00:00   -1.11
                       time  temp_c
0 2022-01-01 00:00:00+00:00    0.11
1 2022-01-01 01:00:00+00:00   -0.18


**Pitfall:** with a default `RangeIndex`, `loc[0:2]` and `iloc[0:2]` differ by one row. After filtering or sorting, the labels are no longer 0..n-1 and `loc[5]` means *label 5*, not *sixth row*.

In [5]:
sub = df[df["temp_c"] < -5]
print(sub.index[:5].tolist())
print("iloc[0] label:", sub.iloc[0].name)
try:
    sub.loc[0]
except KeyError as e:
    print("loc[0] -> KeyError", e)

[244, 674, 723, 724, 725]
iloc[0] label: 244
loc[0] -> KeyError 0


With a `DatetimeIndex`, `loc` accepts partial strings: `df.loc["2022-07"]` is all of July 2022. This is the most convenient way to slice time series.

In [6]:
ts = df.set_index("time")
print(ts.loc["2022-07-01"].shape)                     # one day
print(ts.loc["2022-07"].shape)                        # one month
ts.loc["2022-07-01 06:00":"2022-07-01 09:00", ["consumption_mwh", "price_eur_mwh"]]

(24, 7)
(744, 7)


,consumption_mwh,price_eur_mwh
time,,
2022-07-01 06:00:00+00:00,25537.8,129.20
2022-07-01 07:00:00+00:00,28431.3,132.17
2022-07-01 08:00:00+00:00,29793.0,144.12
2022-07-01 09:00:00+00:00,30331.1,125.46


## 3. Boolean masks

Combine with `&`, `|`, `~` — **not** `and`/`or`/`not` — and wrap each condition in parentheses because `&` binds tighter than `<`.

In [7]:
cold_evening = (df["temp_c"] < 2) & (df["hour"].between(17, 20))
print(cold_evening.sum(), "rows")
df.loc[cold_evening, ["time", "temp_c", "consumption_mwh"]].head(3)

153 rows


,time,temp_c,consumption_mwh
113,2022-01-05 17:00:00+00:00,1.22,38067.2
114,2022-01-05 18:00:00+00:00,-0.02,39331.8
115,2022-01-05 19:00:00+00:00,-0.47,39035.5


`isin` for membership, `between` for closed ranges (both ends inclusive by default), `query` for readable string expressions (`@var` refers to Python variables).

In [8]:
weekend = df[df["dow"].isin([5, 6])]
mild = df[df["temp_c"].between(15, 20)]
thr = 200
spikes = df.query("price_eur_mwh > @thr and hour >= 16")
print(len(weekend), len(mild), len(spikes))
spikes[["time", "price_eur_mwh", "consumption_mwh"]].head(3)

5040 3584 38


,time,price_eur_mwh,consumption_mwh
116,2022-01-05 20:00:00+00:00,320.77,37884.9
1199,2022-02-19 23:00:00+00:00,225.77,26536.7
4818,2022-07-20 18:00:00+00:00,203.91,35084.1


String filters via `.str`: `contains` (regex by default), `startswith`, `len`. Use `na=False` so NaN rows evaluate to False instead of producing NaN in the mask.

In [9]:
meters = pd.read_csv("../data/meters.csv")
print(meters[meters["region"].str.contains("lond", case=False, na=False)].shape)
print(meters[meters["tariff"].str.startswith("F", na=False)].shape)

(99, 7)
(143, 7)


`select_dtypes` picks columns by type — handy to grab all numeric features or all object columns needing clean-up.

In [10]:
print(df.select_dtypes("number").columns.tolist())
print(meters.select_dtypes(include=["object", "bool"]).columns.tolist())

['consumption_mwh', 'temp_c', 'wind_ms', 'solar_wm2', 'price_eur_mwh', 'hour', 'dow']
['meter_id', 'region', 'tariff', 'customer_type', 'signup_date', 'has_solar']


## 4. Renaming and `assign`

`rename(columns={...})` for a few; `df.columns = df.columns.str.lower()` for all. `assign(new=...)` returns a new frame, so it chains — and a lambda inside `assign` sees the intermediate frame.

In [11]:
out = (
    df.rename(columns={"consumption_mwh": "load", "price_eur_mwh": "price"})
      .assign(
          load_gwh=lambda d: d["load"] / 1000,
          revenue_keur=lambda d: d["load"] * d["price"] / 1000,
      )
)
out[["time", "load_gwh", "revenue_keur"]].head(3)

,time,load_gwh,revenue_keur
0,2022-01-01 00:00:00+00:00,26.858,2197.823
1,2022-01-01 01:00:00+00:00,26.178,2309.144
2,2022-01-01 02:00:00+00:00,26.229,2221.892


## 5. Vectorised arithmetic beats `apply`

Column arithmetic runs in C. `apply(lambda row: ...)` with `axis=1` calls Python once per row. Same result, two orders of magnitude slower.

In [12]:
%timeit df["consumption_mwh"] * df["price_eur_mwh"] / 1000

128 µs ± 4.44 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [13]:
%timeit df.apply(lambda r: r["consumption_mwh"] * r["price_eur_mwh"] / 1000, axis=1)

107 ms ± 4.72 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**Interview check:** *"When is `apply` acceptable?"* — when the logic genuinely cannot be vectorised (calling an external function per row) and the frame is small. Otherwise look for `np.where`, `np.select`, `.str`, `.dt`, `map`.

## 6. `map`, `np.where`, `np.select`

`Series.map(dict)` for value → value lookups. `np.where(cond, a, b)` for a two-way conditional; `np.select([conds], [choices], default)` for several.

In [14]:
season_of_month = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring",
                   6: "summer", 7: "summer", 8: "summer", 9: "autumn", 10: "autumn", 11: "autumn"}
df["season"] = df["time"].dt.month.map(season_of_month)
df["is_peak"] = np.where(df["hour"].between(16, 19), 1, 0)
df["load_band"] = np.select(
    [df["consumption_mwh"] < 25_000, df["consumption_mwh"] < 33_000],
    ["low", "mid"],
    default="high",
)
df[["season", "is_peak", "load_band"]].value_counts().head(6)

season  is_peak  load_band
autumn  0        mid          2598
spring  0        mid          2582
winter  0        mid          2378
summer  0        mid          2287
                 low          1393
winter  0        high         1073
Name: count, dtype: int64

## 7. Binning — `cut` (fixed edges) vs `qcut` (quantiles)

`cut` with explicit edges gives interpretable bins (fixed °C). `qcut` gives equal-count bins — good for decile analysis, but the edges move with the data.

In [15]:
df["temp_bin"] = pd.cut(df["temp_c"], bins=[-np.inf, 0, 5, 10, 15, 20, np.inf],
                        labels=["<0", "0-5", "5-10", "10-15", "15-20", ">20"])
df["temp_decile"] = pd.qcut(df["temp_c"], 10, labels=False)
df.groupby("temp_bin", observed=True)["consumption_mwh"].agg(["count", "mean"]).round(0)

,count,mean
temp_bin,,
<0,1144,30060.0
0-5,3671,30962.0
5-10,3985,30363.0
10-15,4070,27431.0
15-20,3578,28219.0
>20,1072,29800.0


## 8. `shift`, `diff`, `pct_change` — mind the sign

`shift(1)` moves values **down**: row *t* now holds the value from *t−1* (the past — safe as a feature). `shift(-1)` pulls *t+1* up to *t* (the future — only ever a target). `diff(k)` is `x - x.shift(k)`; `pct_change` is `x / x.shift(1) - 1`.

In [16]:
s = df["consumption_mwh"]
demo = pd.DataFrame({
    "x": s, "lag1 = shift(1)": s.shift(1), "lead1 = shift(-1)": s.shift(-1),
    "diff1": s.diff(), "pct_change": s.pct_change().round(4),
}).head(4)
demo

,x,lag1 = shift(1),lead1 = shift(-1),diff1,pct_change
0,26858.4,NaN,26177.8,NaN,NaN
1,26177.8,26858.4,26229.4,-680.6,-0.025
2,26229.4,26177.8,25381.3,51.6,0.002
3,25381.3,26229.4,25223.0,-848.1,-0.032


**Interview check:** *"Your feature is `df.x.shift(-24)` — what does that column contain at row t?"* The value 24 hours **ahead**. If it is in `X`, the model is reading the future.

## 9. `rank`, `clip`, `round`

In [17]:
df["price_rank_pct"] = df["price_eur_mwh"].rank(pct=True)
df["price_clipped"] = df["price_eur_mwh"].clip(lower=0, upper=300)
df[["price_eur_mwh", "price_rank_pct", "price_clipped"]].describe().round(2)

,price_eur_mwh,price_rank_pct,price_clipped
count,17520.00,17520.00,17520.00
mean,98.52,0.50,98.51
std,36.90,0.29,36.60
min,-19.94,0.00,0.00
25%,73.49,0.25,73.49
50%,97.68,0.50,97.68
75%,122.81,0.75,122.81
max,419.60,1.00,300.00


## 10. Sorting and extremes

`sort_values` by several keys with per-key direction; `nlargest` / `nsmallest` are faster than sort-then-head; `idxmax` returns the **label** of the max, which you then feed to `loc`.

In [18]:
df.sort_values(["dow", "consumption_mwh"], ascending=[True, False]).head(3)[["time", "dow", "consumption_mwh"]]

,time,dow,consumption_mwh
906,2022-02-07 18:00:00+00:00,0,40824.9
1578,2022-03-07 18:00:00+00:00,0,40408.8
905,2022-02-07 17:00:00+00:00,0,40200.3


In [19]:
print(df.nlargest(3, "price_eur_mwh")[["time", "price_eur_mwh"]])
i = df["consumption_mwh"].idxmax()
print("peak demand hour:", df.loc[i, "time"], df.loc[i, "consumption_mwh"])

                          time  price_eur_mwh
6691 2022-10-06 19:00:00+00:00         419.60
6237 2022-09-17 21:00:00+00:00         379.06
7834 2022-11-23 10:00:00+00:00         375.29
peak demand hour: 2022-02-07 18:00:00+00:00 40824.9


## 11. `set_index` / `reset_index`

Most time-series conveniences (partial-string slicing, `resample`, `asfreq`, alignment) want the time in the index. `reset_index()` puts it back as a column (`drop=True` throws it away).

In [20]:
ts = df.set_index("time").sort_index()
print(ts.index[:2])
back = ts.reset_index()
print(back.columns[:3].tolist())

DatetimeIndex(['2022-01-01 00:00:00+00:00', '2022-01-01 01:00:00+00:00'], dtype='datetime64[ns, UTC]', name='time', freq=None)
['time', 'consumption_mwh', 'temp_c']


## 12. `SettingWithCopyWarning` — what triggers it and the fix

Chained indexing `df[mask]["col"] = value` first creates a (possibly temporary) copy, then assigns into it. The original may or may not change. Pandas warns; **never ignore that warning**. This cell triggers it deliberately.

In [21]:
import warnings
tmp = df[["time", "temp_c"]].copy()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    tmp[tmp["temp_c"] < 0]["temp_c"] = 0        # WRONG: chained assignment
    print("warning raised:", any("SettingWithCopy" in str(x.message) for x in w))
print("values below zero still present:", (tmp["temp_c"] < 0).sum())

warning raised: False
values below zero still present: 1139


Fix: a **single** `.loc[row_mask, col] = value`. And when you take a subset that you will later modify, call `.copy()` explicitly so pandas knows it is yours.

In [22]:
tmp.loc[tmp["temp_c"] < 0, "temp_c"] = 0              # correct
print("values below zero:", (tmp["temp_c"] < 0).sum())

winter = df[df["season"] == "winter"].copy()           # explicit copy -> safe to modify
winter["heating_degrees"] = (15 - winter["temp_c"]).clip(lower=0)
winter[["time", "temp_c", "heating_degrees"]].head(3)

values below zero: 0


,time,temp_c,heating_degrees
0,2022-01-01 00:00:00+00:00,0.11,14.89
1,2022-01-01 01:00:00+00:00,-0.18,15.18
2,2022-01-01 02:00:00+00:00,-1.11,16.11


## 13. `pipe` — functions in a chain

`df.pipe(f, *args)` calls `f(df, *args)`. It keeps a multi-step transformation readable top-to-bottom and each step testable.

In [23]:
def add_time_features(d):
    return d.assign(hour=d["time"].dt.hour, dow=d["time"].dt.dayofweek, month=d["time"].dt.month)

def add_lags(d, col, lags):
    return d.assign(**{f"{col}_lag{k}": d[col].shift(k) for k in lags})

feat = (
    pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
      .sort_values("time")
      .pipe(add_time_features)
      .pipe(add_lags, "consumption_mwh", [1, 24, 168])
)
feat.iloc[168:171, -5:]

,dow,month,consumption_mwh_lag1,consumption_mwh_lag24,consumption_mwh_lag168
168,5,1,31353.0,28476.8,26858.4
169,5,1,28210.5,28227.5,26177.8
170,5,1,26205.3,28457.9,26229.4


## Quick reference

| want | use |
|---|---|
| rows by condition | `df.loc[mask, cols]`, `df.query("...")` |
| rows by position | `df.iloc[i:j]` |
| time window | `ts.loc["2022-07"]` (needs DatetimeIndex) |
| new column from condition | `np.where`, `np.select` |
| lookup table | `s.map(dict)` |
| bins | `pd.cut` (fixed), `pd.qcut` (quantile) |
| past value | `shift(k)` with k > 0 |
| future value (target only) | `shift(-k)` |
| chained transforms | `.assign(...)`, `.pipe(f)` |
| modify a subset | `.loc[mask, col] = v` or `.copy()` first |